# Repository Distribution Analysis
This notebook analyzes the distribution and characteristics of repositories in the dataset.

In [1]:
import pandas as pd
import ast


df_repo = pd.read_csv("../data/repo_characteristics.csv")
df_repo["doc_files"] = df_repo["doc_files"].map(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
df_repo["created_year"] = pd.to_datetime(df_repo["created_at"]).dt.year

## Process Commit Author Information
This section iterates through commit path files to extract unique author information for each commit, handling both GitHub logins and author names.

In [2]:
import json
import tqdm
from pathlib import Path
import pandas as pd

commits_path_dir = Path("../data/commits_path")

records = []
for filepath in tqdm.tqdm(commits_path_dir.iterdir()):
    sha = filepath.name  # markdown sha, no extension
    try:
        with open(filepath, encoding="utf-8") as f:
            commits = json.load(f)
        # collect unique author logins for this markdown file
        authors = set()
        for commit in commits:
            # prefer GitHub login (stable), fallback to author name
            login = (commit.get("author") or {}).get("login")
            if login:
                authors.add(login)
            else:
                name = (commit.get("commit") or {}).get("author", {}).get("name")
                if name:
                    authors.add(name)
        records.append(
            {"sha": sha, "authors": list(authors), "is_login": login is not None}
        )
    except Exception as e:
        print(f"Error reading {filepath}: {e}")

print(
    f"has login: {sum(1 for r in records if r['is_login'])}, no login: {sum(1 for r in records if not r['is_login'])}, ratio: {sum(1 for r in records if r['is_login'])/(len(records)):.1%}"
)
df_commits_path = pd.DataFrame(records)

13360it [00:10, 1258.86it/s]

has login: 11852, no login: 1508, ratio: 88.7%


## Filter Data for Analysis
This section filters the repository and commit data to include only those entries present in the classification dataset, ensuring consistency across analyses.

In [3]:
df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)
df_repo_filtered = df_repo[
    df_repo["full_name"].isin(df_classifications["repository_full_name"])
]
df_commits_filtered = df_commits_path[
    df_commits_path["sha"].isin(df_classifications["sha"])
]

This cell identifies repositories present in the classification data but missing from the repository metadata.

In [4]:
df_classifications[
    ~df_classifications["repository_full_name"].isin(df_repo["full_name"])
]["repository_full_name"].unique()

<StringArray>
[                           'Chailllee/2025Courses_LLM',
                                           'Kcato1/two',
                      'Renan04lima/mapa-do-combustivel',
             'jonathan-nascimento51/glpi_dashboard_cau',
                                'lalalune/ainex-remote',
 'poisontr33s/psychonoir-kontrapunkt-large-file-holder',
               'sethdford/aws-sam-java-personalization']
Length: 7, dtype: str

In [5]:
# Calculates the number of unique authors in the filtered commit data
df_commits_filtered["authors"].explode().nunique()

899

In [6]:
for files in df_repo["doc_files"].to_list():
    filtered_files = [f for f in files if not f.startswith(".specstory/")]
    # print(filtered_files)

In [7]:
s = df_repo["contributors_count"].apply(lambda x: str(x) if x <= 3 else ">=4")
s.value_counts().sort_index().apply(lambda x: f"{x} ({x/len(df_repo)*100:.2f}%)")
# 0 contributors indicates that the commit authors are not linked to GitHub accounts.

contributors_count
0        92 (6.78%)
1      901 (66.45%)
2      225 (16.59%)
3        66 (4.87%)
>=4      72 (5.31%)
Name: count, dtype: str

Cross-language comparison: TS/JS/HTML vs Python

In [ ]:
df_lang = df_classifications.merge(
    df_repo[["full_name", "language"]],
    left_on="repository_full_name",
    right_on="full_name",
    how="left",
)

message_keys = ["sha", "index_in_chat"]
GROUPS = {
    "TS/JS/HTML": {"TypeScript", "JavaScript", "HTML"},
    "Python": {"Python"},
}

results = {}
for group_name, langs in GROUPS.items():
    sub = df_lang[df_lang["language"].isin(langs)]
    n_msgs = sub[message_keys].drop_duplicates().shape[0]
    pct = (
        sub[["main_category", *message_keys]]
        .drop_duplicates()
        .groupby("main_category")
        .size()
        .div(n_msgs)
        .mul(100)
        .sort_index()
    )
    results[group_name] = pct
    print(f"\n{group_name} (n = {n_msgs} messages):")
    for label, v in pct.items():
        print(f"  {label}: {v:.2f}%")

diff = (results["TS/JS/HTML"] - results["Python"]).abs().sort_values(ascending=False)
print("\nAbsolute difference (pp), largest first:")
print(diff.round(2))


TS/JS/HTML (n = 40472 messages):
  1. Code Authoring: 36.34%
  2. Failure Reporting: 25.05%
  3. Inquiry: 18.31%
  4. Context Specification: 14.01%
  5. Validation: 3.46%
  6. Delegation: 15.28%
  7. Workflow Control: 11.22%

Python (n = 13537 messages):
  1. Code Authoring: 27.82%
  2. Failure Reporting: 21.22%
  3. Inquiry: 21.25%
  4. Context Specification: 15.19%
  5. Validation: 5.46%
  6. Delegation: 21.78%
  7. Workflow Control: 12.40%

Absolute difference (pp), largest first:
main_category
1. Code Authoring           8.52
6. Delegation               6.50
2. Failure Reporting        3.84
3. Inquiry                  2.93
5. Validation               2.00
7. Workflow Control         1.19
4. Context Specification    1.17
dtype: float64
